# MGWFER: Causal Spatially Varying Coefficients via Panel Fixed Effects

**Tutorial companion:** [carlos-mendez.org/post/python_mgwrfer/](https://carlos-mendez.org/post/python_mgwrfer/)

A faithful Python replication of [Li & Fotheringham (2026)](https://doi.org/10.1080/24694452.2026.2654481), *"Spatial Context as a Time-Invariant Confounder: A Fixed-Effects Extension of MGWR,"* *Annals of the American Association of Geographers*.

This notebook implements **Multiscale Geographically Weighted Fixed Effects Regression (MGWFER)** — a two-stage local panel estimator that combines (1) the *within-transformation* (which removes time-invariant confounders from panel data) with (2) *Multiscale GWR* (location-specific coefficients at variable-optimal spatial scales). We compare six estimators on a simulated panel (225 units × 3 time periods) with a known unobserved spatial confounder and an active indirect contextual effect channel.

## What this notebook does

1. Installs the **custom mgwr fork** ([GeoZhipengLi/MGWPR](https://github.com/GeoZhipengLi/MGWPR)) that adds panel-data support (`time` parameter) and no-intercept fitting (`constant=False`). The stock pip `mgwr` package will **not** work here.
2. Loads the pre-simulated panel data from this repo's GitHub master branch — no local files required.
3. Runs six estimators: cross-sectional OLS, pooled OLS, individual FE, cross-sectional MGWR, pooled MGWR (PMGWR), and MGWFER (Stage 1 slopes + Stage 2 fixed-effects recovery).
4. Reproduces all eight figures from the blog post: true-coefficient surfaces, PMGWR-vs-MGWFER bias/recovery scatters, coefficient maps, significance maps, bandwidth bar chart, paper Figure 5 (α-surface comparison), and paper Figure 9 (spurious β₄ surfaces).

**Runtime note.** The three MGWR bandwidth searches dominate execution time (~6–10 minutes total on Colab). Each is split into its own cell so you can checkpoint after each fit.


## 1. Setup: install dependencies and clone the custom mgwr fork

The MGWPR fork is a bare module directory (no `setup.py`, no `requirements.txt`), so we clone it to `/content/mgwpr_repo` and prepend it to `sys.path`. Its imports pull `libpysal`, `spreg`, `spglm`, `shapely>=2.0`, `sklearn`, and `tqdm` — most of which are **not** preinstalled on Colab.


In [ ]:
# Install the MGWPR fork's runtime dependencies (sklearn, scipy, numpy already on Colab)
!pip install -q libpysal spreg spglm "shapely>=2.0" tqdm

# Clone the custom fork (idempotent — skipped on re-runs)
![ -d /content/mgwpr_repo ] || git clone -q https://github.com/GeoZhipengLi/MGWPR.git /content/mgwpr_repo

# Prepend the fork to sys.path so `import mgwr` resolves to it (NOT pip's mgwr)
import sys
sys.path.insert(0, "/content/mgwpr_repo")

from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW

print("Custom MGWPR imported successfully")
print(f"  MGWR   from: {MGWR.__module__}")
print(f"  Sel_BW from: {Sel_BW.__module__}")

## 2. Imports and dark-theme styling

Standard scientific Python stack plus `statsmodels` for the OLS / pooled / FE-within baselines and `scipy.stats` for the per-unit t-tests in Stage 2. The dark-navy palette matches the published figures on the blog post.


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.patches import Patch
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Site palette
STEEL_BLUE  = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK  = "#141413"
TEAL        = "#00d4c8"

# Dark theme palette
DARK_NAVY   = "#0f1729"
GRID_LINE   = "#1f2b5e"
LIGHT_TEXT  = "#c8d0e0"
WHITE_TEXT  = "#e8ecf2"

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY,
    "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY,
    "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT,
    "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "axes.grid": True,
    "grid.color": GRID_LINE,
    "grid.linewidth": 0.6,
    "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT,
    "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color": WHITE_TEXT,
    "font.size": 12,
    "legend.frameon": False,
    "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY,
    "savefig.edgecolor": DARK_NAVY,
})

pd.options.display.float_format = "{:.4f}".format

## 3. Load the simulated panel data from GitHub

The pre-simulated dataset lives on the `master` branch of the blog repo. It encodes the paper's data-generating process (Eqs. 39–45) at a 15×15 grid (225 units × 3 time periods = 675 obs) with seed `42` baked in. The CSV ships both the observed panel (`y`, `x1`–`x4`) and the ground-truth coefficient surfaces (`alpha_true`, `beta1_true`–`beta4_true`) on every row, so a single download gives us everything we need.


In [ ]:
BASE_URL = ("https://raw.githubusercontent.com/cmg777/"
            "starter-academic-v501/master/content/post/python_mgwrfer/")

panel_df = pd.read_csv(BASE_URL + "simulated_panel_data.csv")

# Grid metadata recovered directly from the loaded panel
N_TIME  = int(panel_df["time_id"].nunique())
N_UNITS = int(panel_df["unit_id"].nunique())
N_GRID  = int(np.sqrt(N_UNITS))
N_OBS   = len(panel_df)

print(f"Panel shape: {panel_df.shape}")
print(f"N_GRID = {N_GRID}, N_UNITS = {N_UNITS}, N_TIME = {N_TIME}, N_OBS = {N_OBS}")
print()
print(panel_df.head().to_string())

In [ ]:
# Truth tables are time-invariant, so `groupby(unit_id).first()` gives the
# per-unit ground-truth surfaces (one row per spatial location).
truth = panel_df.groupby("unit_id").first()
grid_i = truth["coord_i"].values
grid_j = truth["coord_j"].values

beta_1_true = truth["beta1_true"].values
beta_2_true = truth["beta2_true"].values
beta_3_true = truth["beta3_true"].values
beta_4_true = truth["beta4_true"].values
alpha_true  = truth["alpha_true"].values   # = sc_i in Li & Fotheringham (2026)

# Spatial coordinates for the panel (one row per observation)
coords_panel = list(zip(panel_df["coord_i"].astype(float),
                        panel_df["coord_j"].astype(float)))

print("── True coefficient ranges ──")
for name, vals in [("beta_1 (quadratic)", beta_1_true),
                   ("beta_2 (linear)   ", beta_2_true),
                   ("beta_3 (constant) ", beta_3_true),
                   ("beta_4 (null)     ", beta_4_true),
                   ("alpha   (sc / FE) ", alpha_true)]:
    print(f"  {name}: [{vals.min():.3f}, {vals.max():.3f}], mean={vals.mean():.3f}")

# Verify the indirect contextual channel (paper's bias mechanism) is active
sc_repeat = np.repeat(alpha_true, N_TIME)
print()
print("── Indirect contextual effect strength (paper's bias source) ──")
for k in ["x1", "x2", "x3", "x4"]:
    r = np.corrcoef(panel_df[k].values, sc_repeat)[0, 1]
    print(f"  Cor({k}, sc) = {r:.3f}")
print(f"  Cor(x4, y)  = {np.corrcoef(panel_df['x4'], panel_df['y'])[0,1]:.3f} "
      f"(spurious — beta_4 is zero by construction)")

## 4. True coefficient surfaces

Before fitting anything, visualise the ground truth. The fixed effect $\alpha_i$ (lower-right panel) varies by ~50 units across the grid, while $\beta_1$–$\beta_3$ each vary by at most ~1 unit. Any cross-sectional model that cannot separate $\alpha_i$ from the slopes will see the dominant exponential confounder "leak" into the coefficient surfaces.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
fig.patch.set_facecolor(DARK_NAVY)
fig.suptitle("True Data Generating Process: Spatially Varying Coefficients",
             fontsize=14, color=WHITE_TEXT, y=0.98)

surfaces = [
    (beta_1_true, r"$\beta_1$ (quadratic dome)", "coolwarm"),
    (beta_2_true, r"$\beta_2$ (linear gradient)", "coolwarm"),
    (beta_3_true, r"$\beta_3$ (constant = 1.5)", "coolwarm"),
    (alpha_true,  r"$\alpha_i$ (fixed effect / spatial confounder)", "viridis"),
]
for ax, (vals, title, cmap) in zip(axes.flat, surfaces):
    img = ax.imshow(vals.reshape(N_GRID, N_GRID), cmap=cmap,
                    origin="lower", aspect="equal")
    ax.set_title(title, fontsize=12, color=WHITE_TEXT, pad=8)
    ax.set_xlabel("j (column)", fontsize=10)
    ax.set_ylabel("i (row)", fontsize=10)
    ax.set_xticks([0, N_GRID // 2, N_GRID - 1])
    ax.set_xticklabels(["1", str(N_GRID // 2 + 1), str(N_GRID)])
    ax.set_yticks([0, N_GRID // 2, N_GRID - 1])
    ax.set_yticklabels(["1", str(N_GRID // 2 + 1), str(N_GRID)])
    cbar = plt.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.outline.set_edgecolor(GRID_LINE)
    for lbl in cbar.ax.get_yticklabels():
        lbl.set_color(LIGHT_TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 5. Global baselines (paper Table 2 replication)

Three single-coefficient benchmarks: cross-sectional OLS (period 0 only), pooled OLS (all 675 obs), and the individual fixed-effects (FE) estimator via the within-transformation. None of these models know about location, but together they show that the indirect contextual channel biases naive estimators heavily and that the FE within-transformation neutralises that bias.


In [ ]:
# (a) Cross-sectional OLS on period 0
mask_t0   = panel_df["time_id"].values == 0
y_cs      = panel_df.loc[mask_t0, "y"].values
X_cs_raw  = panel_df.loc[mask_t0, ["x1", "x2", "x3", "x4"]].values
ols_cs    = sm.OLS(y_cs, sm.add_constant(X_cs_raw)).fit()

# (b) Pooled OLS on all 675 obs
X_p_raw   = panel_df[["x1", "x2", "x3", "x4"]].values
ols_pool  = sm.OLS(panel_df["y"].values, sm.add_constant(X_p_raw)).fit()

# (c) Individual FE = within-transformation + OLS (no intercept)
um   = panel_df.groupby("unit_id")[["y", "x1", "x2", "x3", "x4"]].transform("mean")
y_w  = panel_df["y"].values - um["y"].values
X_w  = panel_df[["x1", "x2", "x3", "x4"]].values - um[["x1", "x2", "x3", "x4"]].values
fe_global = sm.OLS(y_w, X_w).fit()

# Recover mean alpha for FE (paper Eq. 19)
y_bar_i = panel_df.groupby("unit_id")["y"].mean().reindex(np.arange(N_UNITS)).values
x_bar_i = (panel_df.groupby("unit_id")[["x1", "x2", "x3", "x4"]]
           .mean().reindex(np.arange(N_UNITS)).values)
alpha_fe_global = y_bar_i - x_bar_i @ fe_global.params

global_tbl = pd.DataFrame({
    "TRUE":         [beta_1_true.mean(), beta_2_true.mean(),
                     beta_3_true.mean(), 0.0,                alpha_true.mean()],
    "OLS_cs":       [ols_cs.params[1], ols_cs.params[2],
                     ols_cs.params[3], ols_cs.params[4],     ols_cs.params[0]],
    "OLS_pool":     [ols_pool.params[1], ols_pool.params[2],
                     ols_pool.params[3], ols_pool.params[4], ols_pool.params[0]],
    "FE_within":    [fe_global.params[0], fe_global.params[1],
                     fe_global.params[2], fe_global.params[3], alpha_fe_global.mean()],
}, index=["beta_1", "beta_2", "beta_3", "beta_4", "mean(alpha)"])

print("── Global model comparison (paper Table 2 replication) ──")
print(global_tbl.to_string())
print()
print(f"FE beta_4 p-value: {fe_global.pvalues[3]:.3g} "
      f"(should be > 0.05 — beta_4 is truly zero)")

The pattern is the paper's headline result on a single screen: OLS and pooled OLS overstate every slope by ~4× and spuriously declare $\beta_4$ highly significant. The individual FE estimator recovers all three true slopes to within 0.07 of truth, returns $\beta_4 \approx 0$ (not significant), and reconstructs the mean of $\alpha_i$ to within 0.06. The within-transformation kills the indirect channel at the global level.


## 6. Pooled MGWR (PMGWR) — the naive local baseline

The simplest local approach ignores the panel structure entirely, treating all 675 observations as independent cross-sectional data and fitting MGWR with an intercept. The custom fork's `time=N_TIME` parameter tells the kernel weighting that observations are grouped in panels of 3 time periods per unit.

**Heads up:** the bandwidth search runs over ~2–4 minutes on Colab.


In [ ]:
Y_raw = panel_df["y"].values.reshape(-1, 1)
X_raw = panel_df[["x1", "x2", "x3", "x4"]].values

# Standardize
Y_std_pooled = (Y_raw - Y_raw.mean()) / Y_raw.std()
X_std_pooled = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
y_mean_pooled, y_std_pooled = float(Y_raw.mean()), float(Y_raw.std())
x_means_pooled = X_raw.mean(axis=0)
x_stds_pooled  = X_raw.std(axis=0)

t0 = time.time()
print("Selecting bandwidths for pooled MGWR...")
pooled_selector = Sel_BW(
    coords_panel, Y_std_pooled, X_std_pooled,
    multi=True, constant=True, time=N_TIME,
)
pooled_bw = pooled_selector.search()
print(f"  PMGWR bandwidths: {pooled_bw}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("Fitting pooled MGWR...")
pooled_model = MGWR(
    coords_panel, Y_std_pooled, X_std_pooled,
    pooled_selector, constant=True, time=N_TIME,
).fit()
print(f"  PMGWR R^2 = {pooled_model.R2:.4f}, AICc = {pooled_model.aicc:.2f}  "
      f"({time.time()-t0:.1f}s)")

In [ ]:
# Back-transform coefficients to original scale
# Note: the PMGWR intercept is reported as (sigma_y * intercept_std), i.e. the
# deviation from the global mean of y — NOT shifted back to absolute level.
# This matches paper Fig. 5's reporting convention.
n_params_pooled = pooled_model.params.shape[1]
pooled_params_orig = np.zeros_like(pooled_model.params)
pooled_params_orig[:, 0] = pooled_model.params[:, 0] * y_std_pooled  # intercept (zero-centred)
for k in range(4):
    pooled_params_orig[:, k + 1] = (pooled_model.params[:, k + 1] *
                                     y_std_pooled / x_stds_pooled[k])

# Average per unit across time
pooled_params_by_unit = np.zeros((N_UNITS, n_params_pooled))
for i in range(N_UNITS):
    mask = panel_df["unit_id"].values == i
    pooled_params_by_unit[i] = pooled_params_orig[mask].mean(axis=0)

pooled_df = pd.DataFrame(
    pooled_params_by_unit,
    columns=["intercept", "beta1_pooled", "beta2_pooled",
             "beta3_pooled", "beta4_pooled"],
)

# Recovery metrics vs truth
rmse_pooled = {}
print("── PMGWR recovery vs truth ──")
for k, (col, true_vals) in enumerate(zip(
    ["beta1_pooled", "beta2_pooled", "beta3_pooled", "beta4_pooled"],
    [beta_1_true, beta_2_true, beta_3_true, beta_4_true],
)):
    rmse = float(np.sqrt(np.mean((pooled_df[col].values - true_vals) ** 2)))
    corr = float(np.corrcoef(pooled_df[col].values, true_vals)[0, 1])
    rmse_pooled[f"beta_{k+1}"] = {"rmse": rmse, "corr": corr}
    print(f"  {col}: RMSE={rmse:.4f}, Corr={corr:.4f}")

alpha_pmgwr = pooled_df["intercept"].values
rmse_alpha_pmgwr = float(np.sqrt(np.mean((alpha_pmgwr - alpha_true) ** 2)))
corr_alpha_pmgwr = float(np.corrcoef(alpha_pmgwr, alpha_true)[0, 1])
print(f"\n  PMGWR intercept (as proxy for sc): "
      f"RMSE={rmse_alpha_pmgwr:.3f}, Corr={corr_alpha_pmgwr:.3f}")

The R² of ~0.99 looks impressive but is misleading — the local intercept absorbs most of `sc_i`'s variation, inflating the fit while the slopes are catastrophically wrong (notably $\beta_1$ is *anti-correlated* with truth). PMGWR offers surfaces, but corrupted ones. MGWFER will give us both unbiased surfaces and a clean recovery of $\alpha_i$.


## 7. Cross-sectional MGWR (one-period baseline)

A standard MGWR fit on a single period (`time_id == 0`). The fork requires the `time` parameter even for cross-sectional data, so we pass `time=1` — making each row a "panel" of length 1, i.e., a true cross-section.

This is the standard "naive local" baseline that the paper compares MGWFER against (paper Table 3).


In [ ]:
mask_t0_full = panel_df["time_id"].values == 0
coords_cs = list(zip(panel_df.loc[mask_t0_full, "coord_i"].astype(float),
                     panel_df.loc[mask_t0_full, "coord_j"].astype(float)))
Y_cs_raw = panel_df.loc[mask_t0_full, "y"].values.reshape(-1, 1)
X_cs_raw_mat = panel_df.loc[mask_t0_full, ["x1", "x2", "x3", "x4"]].values

Y_cs_std = (Y_cs_raw - Y_cs_raw.mean()) / Y_cs_raw.std()
X_cs_std = (X_cs_raw_mat - X_cs_raw_mat.mean(axis=0)) / X_cs_raw_mat.std(axis=0)

y_std_cs_val  = float(Y_cs_raw.std())
y_mean_cs_val = float(Y_cs_raw.mean())
x_stds_cs  = X_cs_raw_mat.std(axis=0)
x_means_cs = X_cs_raw_mat.mean(axis=0)

t0 = time.time()
print("Selecting bandwidths for cross-sectional MGWR (time=1 ⇒ pure cross-section)...")
mgwr_cs_sel = Sel_BW(
    coords_cs, Y_cs_std, X_cs_std,
    multi=True, constant=True, time=1,
)
mgwr_cs_bw = mgwr_cs_sel.search()
print(f"  MGWR_cs bandwidths: {mgwr_cs_bw}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("Fitting MGWR_cs...")
mgwr_cs_model = MGWR(
    coords_cs, Y_cs_std, X_cs_std,
    mgwr_cs_sel, constant=True, time=1,
).fit()
print(f"  MGWR_cs R^2 = {mgwr_cs_model.R2:.4f}, "
      f"AICc = {mgwr_cs_model.aicc:.2f}  ({time.time()-t0:.1f}s)")

In [ ]:
# Back-transform parameters (intercept + 4 slopes)
# Cross-sectional MGWR shifts the intercept back to original location:
#   a = a_centered + y_mean - x_mean @ b
mgwr_cs_params_orig = np.zeros_like(mgwr_cs_model.params)
mgwr_cs_params_orig[:, 0] = mgwr_cs_model.params[:, 0] * y_std_cs_val
for k in range(4):
    mgwr_cs_params_orig[:, k + 1] = (mgwr_cs_model.params[:, k + 1] *
                                      y_std_cs_val / x_stds_cs[k])
mgwr_cs_params_orig[:, 0] = (mgwr_cs_params_orig[:, 0] + y_mean_cs_val
                             - mgwr_cs_params_orig[:, 1:5] @ x_means_cs)

mgwr_cs_df = pd.DataFrame(
    mgwr_cs_params_orig,
    columns=["intercept_mgwr_cs", "beta1_mgwr_cs", "beta2_mgwr_cs",
             "beta3_mgwr_cs", "beta4_mgwr_cs"],
)

rmse_mgwr_cs = {}
print("── MGWR_cs recovery vs truth ──")
for k, (col, true_vals) in enumerate(zip(
    ["beta1_mgwr_cs", "beta2_mgwr_cs", "beta3_mgwr_cs", "beta4_mgwr_cs"],
    [beta_1_true, beta_2_true, beta_3_true, beta_4_true],
)):
    rmse = float(np.sqrt(np.mean((mgwr_cs_df[col].values - true_vals) ** 2)))
    corr = float(np.corrcoef(mgwr_cs_df[col].values, true_vals)[0, 1])
    rmse_mgwr_cs[f"beta_{k+1}"] = {"rmse": rmse, "corr": corr}
    print(f"  {col}: RMSE={rmse:.4f}, Corr={corr:.4f}")

alpha_mgwr_cs = mgwr_cs_df["intercept_mgwr_cs"].values
rmse_alpha_mgwr_cs = float(np.sqrt(np.mean((alpha_mgwr_cs - alpha_true) ** 2)))
corr_alpha_mgwr_cs = float(np.corrcoef(alpha_mgwr_cs, alpha_true)[0, 1])
print(f"\n  MGWR_cs intercept (as proxy for sc): "
      f"RMSE={rmse_alpha_mgwr_cs:.3f}, Corr={corr_alpha_mgwr_cs:.3f}")

## 8. MGWFER Stage 1 — within-transformation + MGWR

Algorithm 1 of Li & Fotheringham (2026), Stage 1: demean by unit (sweeping out the time-invariant confounder), then fit MGWR with `constant=False` on the standardised demeaned variables. Because demeaning removes the intercept by construction, MGWR sees only the within-unit deviations driven by the spatially varying slopes plus noise.

Bandwidth search again runs ~2–4 minutes.


In [ ]:
# Within-transformation: subtract unit means
unit_means = panel_df.groupby("unit_id")[["y", "x1", "x2", "x3", "x4"]].transform("mean")
y_within = (panel_df["y"].values - unit_means["y"].values).reshape(-1, 1)
X_within = np.column_stack([
    panel_df["x1"].values - unit_means["x1"].values,
    panel_df["x2"].values - unit_means["x2"].values,
    panel_df["x3"].values - unit_means["x3"].values,
    panel_df["x4"].values - unit_means["x4"].values,
])

# Sanity check: per-unit mean of y_within should be machine zero
unit_check = pd.DataFrame({"unit_id": panel_df["unit_id"].values,
                          "y_w": y_within.ravel()})
max_unit_mean = unit_check.groupby("unit_id")["y_w"].mean().abs().max()
print(f"y_within range: [{y_within.min():.3f}, {y_within.max():.3f}]")
print(f"Max unit mean after demeaning: {max_unit_mean:.2e}  (should be ~0)")

In [ ]:
# Standardize the demeaned variables and stash the scaling factors
Y_std_fe = (y_within - y_within.mean()) / y_within.std()
X_std_fe = (X_within - X_within.mean(axis=0)) / X_within.std(axis=0)
y_std_fe_val = float(y_within.std())
x_stds_fe    = X_within.std(axis=0)

t0 = time.time()
print("Selecting bandwidths for MGWFER Stage 1 (constant=False)...")
fe_selector = Sel_BW(
    coords_panel, Y_std_fe, X_std_fe,
    multi=True, constant=False, time=N_TIME,
)
fe_bw = fe_selector.search()
print(f"  MGWFER bandwidths: {fe_bw}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("Fitting MGWFER Stage 1...")
fe_model = MGWR(
    coords_panel, Y_std_fe, X_std_fe,
    fe_selector, constant=False, time=N_TIME,
).fit()
print(f"  MGWFER R^2 = {fe_model.R2:.4f}, "
      f"AICc = {fe_model.aicc:.2f}  ({time.time()-t0:.1f}s)")

In [ ]:
# Back-transform slope estimates to the original scale (paper Eq. 29):
#   beta_orig = beta_std * (sigma_Y_dotdot / sigma_X_k_dotdot)
n_params_fe = fe_model.params.shape[1]
fe_params_orig = np.zeros_like(fe_model.params)
for k in range(4):
    fe_params_orig[:, k] = fe_model.params[:, k] * y_std_fe_val / x_stds_fe[k]

# Average per unit across time periods
fe_params_by_unit = np.zeros((N_UNITS, 4))
for i in range(N_UNITS):
    mask = panel_df["unit_id"].values == i
    fe_params_by_unit[i] = fe_params_orig[mask].mean(axis=0)

fe_df = pd.DataFrame(
    fe_params_by_unit,
    columns=["beta1_mgwfer", "beta2_mgwfer", "beta3_mgwfer", "beta4_mgwfer"],
)

rmse_fe = {}
print("── MGWFER Stage 1 recovery vs truth ──")
for k, (col, true_vals) in enumerate(zip(
    ["beta1_mgwfer", "beta2_mgwfer", "beta3_mgwfer", "beta4_mgwfer"],
    [beta_1_true, beta_2_true, beta_3_true, beta_4_true],
)):
    rmse = float(np.sqrt(np.mean((fe_df[col].values - true_vals) ** 2)))
    corr = float(np.corrcoef(fe_df[col].values, true_vals)[0, 1])
    rmse_fe[f"beta_{k+1}"] = {"rmse": rmse, "corr": corr}
    print(f"  {col}: RMSE={rmse:.4f}, Corr={corr:.4f}")

RMSE drops by ~92–96% compared to PMGWR, and the correlation of $\hat\beta_1$ with truth **flips sign** (from −0.46 to +0.82) — the within-transformation has undone the indirect-channel contamination. R² is on the demeaned outcome and isn't directly comparable to PMGWR's R².


## 9. MGWFER Stage 2 — recover individual fixed effects $\hat\alpha_i$

Stage 2 reconstructs the intrinsic contextual effect at each location using paper Eq. 30:

$$\hat\alpha_i = \bar y_i - \sum_k \hat\beta_{bwk}(u_i, v_i) \cdot \bar x_{ik}.$$

We then build per-unit t-tests via paper Eqs. 32–37, with $df = NT - K - N = 675 - 4 - 225 = 446$.


In [ ]:
# Per-unit means
unit_y_mean  = panel_df.groupby("unit_id")["y"].mean().reindex(np.arange(N_UNITS)).values
unit_x_means = (panel_df.groupby("unit_id")[["x1", "x2", "x3", "x4"]]
                .mean().reindex(np.arange(N_UNITS)).values)
beta_unit = fe_params_by_unit                                  # (N_UNITS, 4)

# Eq. 30
alpha_hat = unit_y_mean - np.sum(beta_unit * unit_x_means, axis=1)

# --- Variance computation (paper Eqs. 32-37) ---
# (a) MGWR residual variance on standardised scale (Eq. 34)
resid_std = fe_model.resid_response.ravel()
trS = fe_model.tr_S if hasattr(fe_model, "tr_S") else np.nan
m = N_OBS
sigma_s_sq = (float(np.sum(resid_std ** 2) / (m - trS)) if not np.isnan(trS)
              else float(np.var(resid_std, ddof=4)))

# (b) Rescale to original scale (Eq. 35)
sigma_sq = (N_TIME / (N_TIME - 1)) * (y_std_fe_val ** 2) * sigma_s_sq

# (c) Variance of per-unit slopes (Eqs. 36-37), diagonal approximation
CCT = fe_model.CCT                                              # (NT, K) standardised
CCT_by_unit = np.zeros((N_UNITS, CCT.shape[1]))
for i in range(N_UNITS):
    mask = panel_df["unit_id"].values == i
    CCT_by_unit[i] = CCT[mask].mean(axis=0)

S_beta = y_std_fe_val / x_stds_fe                              # (K,)
var_beta_unit = (CCT_by_unit * sigma_s_sq) * (S_beta ** 2)     # (N_UNITS, K)

# (d) Var[alpha_i] = sigma_sq / T + sum_k x_bar_ik^2 * var_beta_ik
var_alpha = (sigma_sq / N_TIME) + np.sum(unit_x_means ** 2 * var_beta_unit, axis=1)
se_alpha  = np.sqrt(np.clip(var_alpha, a_min=1e-12, a_max=None))

# (e) per-unit t-test
t_alpha   = alpha_hat / se_alpha
df_alpha  = N_OBS - 4 - N_UNITS                                # 446
p_alpha   = 2 * (1 - stats.t.cdf(np.abs(t_alpha), df=df_alpha))
sig5      = p_alpha < 0.05

rmse_alpha = float(np.sqrt(np.mean((alpha_hat - alpha_true) ** 2)))
corr_alpha = float(np.corrcoef(alpha_hat, alpha_true)[0, 1])

print("── MGWFER Stage 2 recovery vs truth ──")
print(f"  alpha_hat:  range [{alpha_hat.min():.3f}, {alpha_hat.max():.3f}], "
      f"mean = {alpha_hat.mean():.3f}")
print(f"  true alpha: range [{alpha_true.min():.3f}, {alpha_true.max():.3f}], "
      f"mean = {alpha_true.mean():.3f}")
print(f"  alpha_hat recovery: RMSE = {rmse_alpha:.4f}, Corr = {corr_alpha:.4f}")
print(f"  Significant at 5%: {int(sig5.sum())}/{N_UNITS} units "
      f"({100 * sig5.mean():.1f}%)   df = {df_alpha}")

## 10. Model comparison (paper Table 3 replication)

The headline summary: MGWFER reduces RMSE by an order of magnitude across every slope coefficient *and* recovers the intrinsic contextual effect with Pearson correlation ≈ 1.0.


In [ ]:
rows = []
for k in range(1, 5):
    key = f"beta_{k}"
    rows.append({"metric": f"RMSE_beta_{k}",
                 "MGWR_cs": rmse_mgwr_cs[key]["rmse"],
                 "PMGWR":   rmse_pooled[key]["rmse"],
                 "MGWFER":  rmse_fe[key]["rmse"]})
for k in range(1, 5):
    key = f"beta_{k}"
    rows.append({"metric": f"Corr_beta_{k}",
                 "MGWR_cs": rmse_mgwr_cs[key]["corr"],
                 "PMGWR":   rmse_pooled[key]["corr"],
                 "MGWFER":  rmse_fe[key]["corr"]})
rows.append({"metric": "R_squared",
             "MGWR_cs": mgwr_cs_model.R2,
             "PMGWR":   pooled_model.R2,
             "MGWFER":  fe_model.R2})
rows.append({"metric": "AICc",
             "MGWR_cs": mgwr_cs_model.aicc,
             "PMGWR":   pooled_model.aicc,
             "MGWFER":  fe_model.aicc})
rows.append({"metric": "RMSE_alpha",
             "MGWR_cs": rmse_alpha_mgwr_cs,
             "PMGWR":   rmse_alpha_pmgwr,
             "MGWFER":  rmse_alpha})
rows.append({"metric": "Corr_alpha",
             "MGWR_cs": corr_alpha_mgwr_cs,
             "PMGWR":   corr_alpha_pmgwr,
             "MGWFER":  corr_alpha})

comp_df = pd.DataFrame(rows).set_index("metric")
print("── Local model comparison (paper Table 3 replication) ──")
print(comp_df.to_string())

## 11. Coefficient recovery scatters

Two 1×3 scatter panels comparing true vs estimated coefficients for PMGWR (the biased baseline) and MGWFER (the bias-corrected estimator). In a perfect estimator, all points would lie on the dashed 45-degree line.


In [ ]:
# Figure: PMGWR bias
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor(DARK_NAVY)
fig.suptitle("Pooled MGWR (PMGWR): Biased Estimates (ignoring fixed effects)",
             fontsize=13, color=WHITE_TEXT, y=1.02)

true_arrays   = [beta_1_true, beta_2_true, beta_3_true]
pooled_arrays = [pooled_df["beta1_pooled"].values,
                 pooled_df["beta2_pooled"].values,
                 pooled_df["beta3_pooled"].values]
labels = [r"$\beta_1$ (quadratic)", r"$\beta_2$ (linear)",
          r"$\beta_3$ (constant)"]

for ax, true_vals, est_vals, label in zip(axes, true_arrays, pooled_arrays, labels):
    ax.scatter(true_vals, est_vals, color=STEEL_BLUE, alpha=0.4, s=15, zorder=3)
    lims = [min(true_vals.min(), est_vals.min()) - 0.2,
            max(true_vals.max(), est_vals.max()) + 0.2]
    ax.plot(lims, lims, color=WARM_ORANGE, linewidth=2, linestyle="--", zorder=2)
    rmse = np.sqrt(np.mean((est_vals - true_vals) ** 2))
    corr = np.corrcoef(true_vals, est_vals)[0, 1]
    ax.text(0.05, 0.95, f"RMSE = {rmse:.3f}\nCorr = {corr:.3f}",
            transform=ax.transAxes, fontsize=10, verticalalignment="top",
            bbox=dict(facecolor="#1a1a2e", edgecolor=LIGHT_TEXT, alpha=0.9),
            color=WHITE_TEXT)
    ax.set_xlabel("True coefficient", fontsize=10)
    ax.set_ylabel("Pooled MGWR estimate", fontsize=10)
    ax.set_title(label, fontsize=11, color=WHITE_TEXT)
    ax.set_xlim(lims); ax.set_ylim(lims)

plt.tight_layout()
plt.show()

In [ ]:
# Figure: MGWFER recovery
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor(DARK_NAVY)
fig.suptitle("MGWFER: Bias-Corrected Estimates (fixed effects removed)",
             fontsize=13, color=WHITE_TEXT, y=1.02)

fe_arrays = [fe_df["beta1_mgwfer"].values,
             fe_df["beta2_mgwfer"].values,
             fe_df["beta3_mgwfer"].values]

for ax, true_vals, est_vals, label in zip(axes, true_arrays, fe_arrays, labels):
    ax.scatter(true_vals, est_vals, color=TEAL, alpha=0.4, s=15, zorder=3)
    lims = [min(true_vals.min(), est_vals.min()) - 0.2,
            max(true_vals.max(), est_vals.max()) + 0.2]
    ax.plot(lims, lims, color=WARM_ORANGE, linewidth=2, linestyle="--", zorder=2)
    rmse = np.sqrt(np.mean((est_vals - true_vals) ** 2))
    corr = np.corrcoef(true_vals, est_vals)[0, 1]
    ax.text(0.05, 0.95, f"RMSE = {rmse:.3f}\nCorr = {corr:.3f}",
            transform=ax.transAxes, fontsize=10, verticalalignment="top",
            bbox=dict(facecolor="#1a1a2e", edgecolor=LIGHT_TEXT, alpha=0.9),
            color=WHITE_TEXT)
    ax.set_xlabel("True coefficient", fontsize=10)
    ax.set_ylabel("MGWFER estimate", fontsize=10)
    ax.set_title(label, fontsize=11, color=WHITE_TEXT)
    ax.set_xlim(lims); ax.set_ylim(lims)

plt.tight_layout()
plt.show()

## 12. Spatial coefficient maps — true vs MGWFER

A 2×3 grid: top row is the ground truth, bottom row is MGWFER's recovered surface for each of the three causally active coefficients. Bandwidth is annotated under each MGWFER panel.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor(DARK_NAVY)
fig.suptitle("Spatial Coefficient Recovery: True (top) vs MGWFER (bottom)",
             fontsize=14, color=WHITE_TEXT, y=0.99)

# Row 1: true
true_titles = [r"True $\beta_1$", r"True $\beta_2$", r"True $\beta_3$"]
for col_idx, (vals, title) in enumerate(zip(true_arrays, true_titles)):
    ax = axes[0, col_idx]
    img = ax.imshow(vals.reshape(N_GRID, N_GRID), cmap="coolwarm",
                    origin="lower", aspect="equal")
    ax.set_title(title, fontsize=11, color=WHITE_TEXT, pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    cbar = plt.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.outline.set_edgecolor(GRID_LINE)
    for lbl in cbar.ax.get_yticklabels():
        lbl.set_color(LIGHT_TEXT)

# Row 2: MGWFER
bw_labels = list(fe_bw) if hasattr(fe_bw, "__len__") else [fe_bw] * 4
fe_titles = [r"MGWFER $\hat\beta_1$", r"MGWFER $\hat\beta_2$",
             r"MGWFER $\hat\beta_3$"]
for col_idx, (vals, title, bw) in enumerate(zip(fe_arrays, fe_titles, bw_labels[:3])):
    ax = axes[1, col_idx]
    vmin = true_arrays[col_idx].min()
    vmax = true_arrays[col_idx].max()
    img = ax.imshow(vals.reshape(N_GRID, N_GRID), cmap="coolwarm",
                    origin="lower", aspect="equal", vmin=vmin, vmax=vmax)
    bw_val = int(bw) if not np.isnan(bw) else "?"
    ax.set_title(f"{title} (bw={bw_val})", fontsize=11, color=WHITE_TEXT, pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    cbar = plt.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.outline.set_edgecolor(GRID_LINE)
    for lbl in cbar.ax.get_yticklabels():
        lbl.set_color(LIGHT_TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 13. Significance maps

Filtered t-values (corrected for multiple testing across the 225 spatial units, following da Silva & Fotheringham 2016) for each coefficient. Orange = significantly positive, blue = significantly negative, dark = not significant.


In [ ]:
fe_tvalues = fe_model.filter_tvals()

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
fig.patch.set_facecolor(DARK_NAVY)
fig.suptitle("MGWFER: Statistical Significance of Spatially Varying Coefficients",
             fontsize=14, color=WHITE_TEXT, y=0.98)

coef_names = [r"$\beta_1$ (quadratic)", r"$\beta_2$ (linear)",
              r"$\beta_3$ (constant)", r"$\beta_4$ (null)"]

def hex2rgb(h):
    return np.array([int(h[1:3], 16), int(h[3:5], 16), int(h[5:7], 16)]) / 255

orange_rgb = hex2rgb(WARM_ORANGE)
blue_rgb   = hex2rgb(STEEL_BLUE)
grid_rgb   = hex2rgb(GRID_LINE)

sig_summary = {}
for col_idx, (ax, name) in enumerate(zip(axes.flat, coef_names)):
    t_col = fe_tvalues[:, col_idx]
    t_by_unit = np.zeros(N_UNITS)
    for i in range(N_UNITS):
        mask = panel_df["unit_id"].values == i
        t_by_unit[i] = t_col[mask].mean()

    sig_pos = t_by_unit > 0
    sig_neg = t_by_unit < 0
    not_sig = t_by_unit == 0
    n_pos, n_neg, n_ns = int(sig_pos.sum()), int(sig_neg.sum()), int(not_sig.sum())
    sig_summary[name] = {"positive": n_pos, "negative": n_neg, "not_sig": n_ns}

    color_grid = np.zeros((N_UNITS, 3))
    color_grid[sig_pos] = orange_rgb
    color_grid[sig_neg] = blue_rgb
    color_grid[not_sig] = grid_rgb

    ax.imshow(color_grid.reshape(N_GRID, N_GRID, 3),
              origin="lower", aspect="equal")
    ax.set_title(name, fontsize=11, color=WHITE_TEXT, pad=6)
    ax.set_xticks([]); ax.set_yticks([])

    handles = [
        Patch(facecolor=WARM_ORANGE, label=f"Sig. positive (n={n_pos})"),
        Patch(facecolor=GRID_LINE,   label=f"Not significant (n={n_ns})"),
        Patch(facecolor=STEEL_BLUE,  label=f"Sig. negative (n={n_neg})"),
    ]
    leg = ax.legend(handles=handles, loc="lower left", fontsize=9)
    leg.get_frame().set_facecolor("#1a1a2e")
    leg.get_frame().set_edgecolor(LIGHT_TEXT)
    leg.get_frame().set_alpha(0.9)
    for txt in leg.get_texts():
        txt.set_color(WHITE_TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("── Significance summary ──")
for name, counts in sig_summary.items():
    print(f"  {name}: positive={counts['positive']}, "
          f"not_sig={counts['not_sig']}, negative={counts['negative']}")

## 14. Bandwidth comparison

Bandwidths reveal *how* each estimator reads the spatial structure. PMGWR collapses everything into a narrow range (because, under the indirect channel, every covariate looks like a noisy proxy for `sc`). MGWFER alone returns bandwidths that match the true process scales — small for $\beta_1$ (local dome), large for $\beta_3$ (spatially constant).


In [ ]:
mgwr_cs_bw_list = list(mgwr_cs_bw) if hasattr(mgwr_cs_bw, "__len__") else [mgwr_cs_bw]
pooled_bw_list   = list(pooled_bw)   if hasattr(pooled_bw,   "__len__") else [pooled_bw]
fe_bw_list       = list(fe_bw)       if hasattr(fe_bw,       "__len__") else [fe_bw]

# MGWR_cs and PMGWR include intercept at index 0; drop it for x1-x4 comparison
mgwr_cs_var_bw = mgwr_cs_bw_list[1:5] if len(mgwr_cs_bw_list) >= 5 else mgwr_cs_bw_list
pooled_var_bw  = pooled_bw_list[1:5]  if len(pooled_bw_list)  >= 5 else pooled_bw_list
fe_var_bw      = fe_bw_list[:4]

print(f"  MGWR_cs bws (x1-x4): {[int(b) for b in mgwr_cs_var_bw]}")
print(f"  PMGWR   bws (x1-x4): {[int(b) for b in pooled_var_bw]}")
print(f"  MGWFER  bws (x1-x4): {[int(b) for b in fe_var_bw]}")

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(DARK_NAVY)

x_pos = np.arange(4)
w = 0.27
bars0 = ax.bar(x_pos - w, mgwr_cs_var_bw, w, color=STEEL_BLUE,
               alpha=0.85, label="MGWR (cross-section)")
bars1 = ax.bar(x_pos,     pooled_var_bw,  w, color=WARM_ORANGE,
               alpha=0.85, label="PMGWR (pooled)")
bars2 = ax.bar(x_pos + w, fe_var_bw,      w, color=TEAL,
               alpha=0.85, label="MGWFER")

for bars, color in [(bars0, STEEL_BLUE), (bars1, WARM_ORANGE), (bars2, TEAL)]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f"{int(bar.get_height())}",
                ha="center", fontsize=9, color=color)

ax.set_xlabel("Covariate", fontsize=11)
ax.set_ylabel("Bandwidth (nearest neighbors)", fontsize=11)
ax.set_title("Bandwidth Comparison: MGWR_cs vs PMGWR vs MGWFER", fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels([r"$x_1$ (quadratic)", r"$x_2$ (linear)",
                    r"$x_3$ (constant)", r"$x_4$ (null)"])
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 15. Spatial-context surface — paper Figure 5 replication

A 2×2 grid comparing the true spatial-context surface $sc_i$ against each local model's estimate. MGWFER's $\hat\alpha_i$ should be nearly indistinguishable from the truth; MGWR_cs's and PMGWR's local intercepts should show compressed ranges and (in PMGWR's case) inverted signs.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.patch.set_facecolor(DARK_NAVY)
fig.suptitle("Spatial context surface — recovered by each model "
             r"(true $sc_i$ vs $\hat\alpha_i$)",
             fontsize=14, color=WHITE_TEXT, y=0.99)

vmin = float(alpha_true.min())
vmax = float(alpha_true.max())

alpha_panels = [
    (alpha_true,    r"True $sc_i$ (DGP)",                            None, None),
    (alpha_hat,     r"MGWFER $\hat\alpha_i$ (Stage 2, Eq. 30)",    rmse_alpha,        corr_alpha),
    (alpha_mgwr_cs, r"MGWR (cross-sectional) — local intercept",      rmse_alpha_mgwr_cs, corr_alpha_mgwr_cs),
    (alpha_pmgwr,   r"PMGWR (pooled) — local intercept",              rmse_alpha_pmgwr,   corr_alpha_pmgwr),
]

for ax, (vals, title, rmse_v, corr_v) in zip(axes.flat, alpha_panels):
    img = ax.imshow(vals.reshape(N_GRID, N_GRID), cmap="viridis",
                    origin="lower", aspect="equal", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=12, color=WHITE_TEXT, pad=8)
    ax.set_xlabel("j (column)", fontsize=10)
    ax.set_ylabel("i (row)", fontsize=10)
    ax.set_xticks([0, N_GRID // 2, N_GRID - 1])
    ax.set_xticklabels(["1", str(N_GRID // 2 + 1), str(N_GRID)])
    ax.set_yticks([0, N_GRID // 2, N_GRID - 1])
    ax.set_yticklabels(["1", str(N_GRID // 2 + 1), str(N_GRID)])
    cbar = plt.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.outline.set_edgecolor(GRID_LINE)
    for lbl in cbar.ax.get_yticklabels():
        lbl.set_color(LIGHT_TEXT)

    range_str = f"range: [{vals.min():.1f}, {vals.max():.1f}]"
    if rmse_v is not None:
        ann = f"{range_str}\nRMSE = {rmse_v:.2f}\nCorr = {corr_v:.3f}"
    else:
        ann = range_str
    ax.text(0.02, 0.98, ann,
            transform=ax.transAxes, fontsize=9, verticalalignment="top",
            bbox=dict(facecolor="#1a1a2e", edgecolor=LIGHT_TEXT, alpha=0.9),
            color=WHITE_TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 16. Spurious $\beta_4$ surface — paper Figure 9 replication

Since `x4` has zero causal effect on `y` but is correlated with `sc` through the indirect channel, MGWR_cs and PMGWR pick up a column-aligned spurious "effect" that tracks the horizontal `sc` gradient. MGWFER's $\hat\beta_4$ surface should be near-zero and structureless.


In [ ]:
fig, axes_b4 = plt.subplots(1, 3, figsize=(15, 5.5))
fig.patch.set_facecolor(DARK_NAVY)
fig.suptitle(r"Spurious $\hat\beta_4$ surface — true $\beta_4 \equiv 0$ "
             "(paper Fig. 9 replication)",
             fontsize=14, color=WHITE_TEXT, y=1.02)

b4_panels = [
    (mgwr_cs_df["beta4_mgwr_cs"].values, "MGWR (cross-sectional)"),
    (pooled_df["beta4_pooled"].values,    "PMGWR (pooled)"),
    (fe_df["beta4_mgwfer"].values,        "MGWFER (within + MGWR)"),
]

b4_min = min(v.min() for v, _ in b4_panels)
b4_max = max(v.max() for v, _ in b4_panels)
b4_lim = max(abs(b4_min), abs(b4_max), 0.1)

for ax, (vals, title) in zip(axes_b4, b4_panels):
    img = ax.imshow(vals.reshape(N_GRID, N_GRID), cmap="coolwarm",
                    origin="lower", aspect="equal", vmin=-b4_lim, vmax=b4_lim)
    ax.set_title(title, fontsize=12, color=WHITE_TEXT, pad=8)
    ax.set_xlabel("j (column)", fontsize=10)
    ax.set_ylabel("i (row)", fontsize=10)
    ax.set_xticks([0, N_GRID // 2, N_GRID - 1])
    ax.set_xticklabels(["1", str(N_GRID // 2 + 1), str(N_GRID)])
    ax.set_yticks([0, N_GRID // 2, N_GRID - 1])
    ax.set_yticklabels(["1", str(N_GRID // 2 + 1), str(N_GRID)])
    cbar = plt.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.outline.set_edgecolor(GRID_LINE)
    for lbl in cbar.ax.get_yticklabels():
        lbl.set_color(LIGHT_TEXT)
    rmse_b4 = float(np.sqrt(np.mean(vals ** 2)))
    ax.text(0.02, 0.98,
            f"range: [{vals.min():.2f}, {vals.max():.2f}]\nRMSE vs 0 = {rmse_b4:.3f}",
            transform=ax.transAxes, fontsize=9, verticalalignment="top",
            bbox=dict(facecolor="#1a1a2e", edgecolor=LIGHT_TEXT, alpha=0.9),
            color=WHITE_TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 17. Summary

| Result | Where to look |
|--------|---------------|
| Global Table 2 (OLS biased, FE corrects) | Section 5 |
| MGWFER reduces local RMSE by 92–96% vs PMGWR | Section 10 |
| MGWFER recovers $\hat\alpha_i$ at Corr ≈ 1.0 vs truth | Section 9 |
| Paper Fig. 5 (α-surface comparison) | Section 15 |
| Paper Fig. 9 (spurious $\beta_4$ surfaces) | Section 16 |
| $\hat\beta_1$ correlation flips from −0.46 (PMGWR) to +0.82 (MGWFER) | Section 8 |
| PMGWR bandwidths collapse to 44–50; MGWFER recovers 50–116 | Section 14 |

**The mechanism is the within-transformation.** Demeaning by unit removes the time-invariant part of `sc` from both `y` and the `x_k`'s, severing the `sc → x_k` backdoor path. Everything downstream — standardisation, bandwidth search, t-tests — is contingent on this single move.

### Identification assumptions

A causal reading of MGWFER coefficients depends on four assumptions (Li & Fotheringham 2026):

1. **Time-invariant spatial context.** $\alpha_i$ does not change over the study period.
2. **Strict exogeneity given the fixed effects.** Conditional on $\alpha_i$ and observed $X_{it}$'s, errors are uncorrelated with covariates in all periods.
3. **No time-varying unobserved confounders.** The within-transformation deals with time-invariant confounding only.
4. **Parameter stability over time.** Slopes $\beta_{bwk}(u_i, v_i)$ are constant across the $T$ periods.

### Exercises

1. Re-run the bandwidth search with `N_TIME = 10` (you would need to regenerate the panel from the seed — see `script.py` for the DGP). How does the bias–variance tradeoff change?
2. Try a weaker indirect channel by simulating with `SC_COUPLING = 0.02`. Where does PMGWR become "good enough"?
3. Apply MGWFER to a real panel (e.g., US counties on ACS data) and compare the $\hat\alpha_i$ surface to what cross-sectional MGWR produces.

### References

- [Li, Z. & Fotheringham, A.S. (2026). Spatial Context as a Time-Invariant Confounder: A Fixed-Effects Extension of MGWR. *Annals of the AAG*.](https://doi.org/10.1080/24694452.2026.2654481)
- [Fotheringham, A.S., Yang, W. & Kang, W. (2017). Multiscale GWR. *Annals of the AAG*, 107(6).](https://doi.org/10.1080/24694452.2017.1352480)
- [Oshan, T. et al. (2019). mgwr: A Python Implementation of MGWR. *JOSS* 4(42).](https://doi.org/10.21105/joss.01823)
- [da Silva, A.R. & Fotheringham, A.S. (2016). The multiple testing issue in GWR. *Geographical Analysis* 48(3).](https://doi.org/10.1111/gean.12084)
- [GeoZhipengLi/MGWPR — custom mgwr fork with panel data support](https://github.com/GeoZhipengLi/MGWPR)
- Wooldridge, J.M. (2010). *Econometric Analysis of Cross Section and Panel Data*, 2nd ed. MIT Press.
